# Tracing a framework agent — LangChain + one-line autolog

*Part of the `gen_ai/` track. Prerequisites: `a_tracing_quickstart`, `b_tracing_a_multistep_app`.*

## The problem: an agent's control flow is decided at *runtime*

`b_tracing_a_multistep_app` traced a **fixed** pipeline — *you* wrote `retrieve()` then `generate()`. An **agent** is different: the LLM itself decides, turn by turn, *which* tool to call and *when to stop*. The control flow isn't in your code — it emerges as the model runs. When an agent misbehaves (calls the wrong tool, loops forever, ignores a result), a flat log is useless.

Two things make this notebook new:

- **A real framework.** We build the agent with **LangChain** instead of by hand. Frameworks are how people actually build agents and RAG.
- **One-line tracing.** `mlflow.langchain.autolog()` records a span for every LLM decision and every tool call — automatically. The spans you wrote by hand in `b_`, now generated for a far more dynamic app, for free.

## Prerequisites

- **Server on port 5001** (`mlflow server --host 127.0.0.1 --port 5001` from `src/`).
- **Ollama** serving a **tool-capable** model — `qwen3:8b` works. (Not every model supports tool-calling; small or older ones may not.)
- **LangChain**, added to the project:

  ```bash
  uv add langchain langchain-openai langgraph
  ```

**Diverges from upstream tutorial:** the LangChain API moves fast — this uses the **v1** entrypoint `langchain.agents.create_agent`. Older tutorials use `initialize_agent` / `AgentExecutor`, which are superseded. Pinning the version (via `uv.lock`) is how this repo stays reproducible against a fast-moving dependency.

In [ ]:
import mlflow
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("genai-langchain-agent")
mlflow.langchain.autolog()  # one line: trace every LLM step and tool call below

# The agent's LLM. Local Ollama is the zero-cost default.
llm = ChatOpenAI(model="qwen3:8b", base_url="http://localhost:11434/v1",
                 api_key="ollama", temperature=0,
                 timeout=60, max_retries=0, max_tokens=512)

# Reliable tool-calling matters for agents; a hosted model is faster and steadier.
# To use Azure instead, set the .env vars from c_genai_evaluation and swap in:
#   from langchain_openai import AzureChatOpenAI
#   import os; from dotenv import load_dotenv, find_dotenv; load_dotenv(find_dotenv())
#   llm = AzureChatOpenAI(
#       azure_endpoint=os.environ['AZURE_OPENAI_BASE_URL'].removesuffix('/openai'),
#       api_key=os.environ['AZURE_OPENAI_API_KEY'],
#       api_version=os.environ.get('AZURE_OPENAI_API_VERSION', '2024-10-21'),
#       azure_deployment='gpt-4.1', temperature=0)

## Give the agent some tools

A `@tool` is just a Python function with a docstring — the agent reads the docstring to decide when to use it. Two tiny math tools are enough to force a **two-step** plan.

In [ ]:
@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

## Build and run the agent

`create_agent` wires the LLM to the tools and runs the loop: the model calls a tool, sees the result, decides what to do next, and stops when it has the answer. We never write that loop — and `autolog` records all of it.

The `/no_think` keeps `qwen3` fast (see the thinking-mode note in `a_tracing_quickstart`); a non-reasoning model ignores it.

In [ ]:
agent = create_agent(llm, tools=[add, multiply])

result = agent.invoke({
    "messages": [{"role": "user",
                  "content": "What is (3 + 4) multiplied by 5? Use the tools step by step. /no_think"}]
})
print(result["messages"][-1].content)

## Read the agent trace

Open <http://127.0.0.1:5001> → **Traces**. The newest trace is the agent's whole run, and it captured a loop *you never wrote*:

```
agent
├─ ChatOpenAI      — model decides: call add(3, 4)
├─ add             — tool returns 7
├─ ChatOpenAI      — model decides: call multiply(7, 5)
├─ multiply        — tool returns 35
└─ ChatOpenAI      — model writes the final answer
```

Every decision and tool result is there, with inputs and outputs. If the agent had multiplied before adding, or called a tool with the wrong arguments, you'd see exactly that step — which is the only practical way to debug agents.

## The payoff vs. `b_`

In `b_` you wrote `@mlflow.trace` on each function yourself, on a pipeline whose steps you controlled. Here a **single** `mlflow.langchain.autolog()` traced a **more complex, runtime-decided** agent — with no tracing code in the app at all. That's the deal frameworks offer, and MLflow's autolog meets them at the door: the same span model, whether you instrument by hand (`b_`) or let the framework emit them (here).

The same one-liner exists for other stacks — `mlflow.llama_index.autolog()`, `mlflow.dspy.autolog()`, and more — so the lesson transfers beyond LangChain.

## Next steps

- **`c_genai_evaluation.ipynb`** (if you skipped it) — score this agent's answers with LLM-as-judge; `predict_fn` can wrap `agent.invoke`.
- **`f_genai_app_serving.ipynb`** (planned) — log this agent as an MLflow **model** (models-from-code) and serve it over REST, the GenAI parallel of `ml/g_model_serving`.